In [2]:
# %%
import math
import bisect
import gc
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import chess

BATCH_SIZE = 256
VAL_FRAC   = 0.1
CKPT_PATH  = "../weights/new_labels2.pth"
in_ch = 19

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)




Using device: cuda


In [3]:

import json
import math
import bisect
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset

def move_to_index(move: chess.Move) -> int:
    if move.promotion and move.promotion != chess.QUEEN:
        promo_offset = {
            chess.KNIGHT: 0,
            chess.BISHOP: 1,
            chess.ROOK: 2,
        }
        return 4096 + move.from_square * 3 + promo_offset[move.promotion]
    return move.from_square * 64 + move.to_square


def get_legal_mask(board: chess.Board) -> np.ndarray:
    mask = np.zeros(4672, dtype=np.bool_)
    for move in board.legal_moves:
        mask[move_to_index(move)] = True
    return mask






class FastChessShardDataset(Dataset):
    def __init__(
        self,
        manifest_path,
        flatten: bool = False,
        cp_clip: float = 3000.0,
        sigmoid_scale: float = 400.0,
        mate_cp: float = 10000.0,
        use_sigmoid: bool = True,
    ):
        manifest_path = Path(manifest_path)
        with open(manifest_path, "r", encoding="utf-8") as f:
            manifest = json.load(f)

        self.shards = manifest["shards"]
        self.lengths = [int(s["length"]) for s in self.shards]
        self.cum_lengths = np.cumsum(self.lengths).tolist()
        self.N = int(sum(self.lengths))

        self.flatten = flatten
        self.cp_clip = float(cp_clip)
        self.sigmoid_scale = float(sigmoid_scale)
        self.mate_cp = float(mate_cp)
        self.use_sigmoid = use_sigmoid

        self._current_shard_idx = None
        self._positions = None
        self._cp_raw = None
        self._is_mate = None
        self._mate_in = None
        self._policy_index = None
        self._legal_mask = None

    def __len__(self):
        return self.N

    def _locate(self, idx):
        shard_idx = bisect.bisect_right(self.cum_lengths, idx)
        start = 0 if shard_idx == 0 else self.cum_lengths[shard_idx - 1]
        local_idx = idx - start
        return shard_idx, local_idx

    def _open_shard(self, shard_idx):
        if self._current_shard_idx == shard_idx:
            return

        shard_dir = Path(self.shards[shard_idx]["dir"])

        self._positions = np.load(shard_dir / "positions.npy", mmap_mode="r")
        self._cp_raw = np.load(shard_dir / "evaluations_cp_raw.npy", mmap_mode="r")
        self._is_mate = np.load(shard_dir / "is_mate.npy", mmap_mode="r")
        self._mate_in = np.load(shard_dir / "mate_in.npy", mmap_mode="r")
        self._policy_index = np.load(shard_dir / "policy_index.npy", mmap_mode="r")
        self._legal_mask = np.load(shard_dir / "legal_mask.npy", mmap_mode="r")

        self._current_shard_idx = shard_idx

    def _raw_label_to_cp(self, cp_raw, is_mate, mate_in):
        if int(is_mate) == 1:
            if mate_in > 0:
                return self.mate_cp
            if mate_in < 0:
                return -self.mate_cp
            return 0.0
        return float(cp_raw)

    def _cp_to_target(self, cp):
        cp = np.clip(cp, -self.cp_clip, self.cp_clip)
        if self.use_sigmoid:
            return 1.0 / (1.0 + math.exp(-cp / self.sigmoid_scale))
        return cp

    def __getitem__(self, idx):
        shard_idx, local_idx = self._locate(idx)
        self._open_shard(shard_idx)

        x = self._positions[local_idx].astype(np.float32, copy=False)
        if self.flatten:
            x = x.reshape(-1)

        cp = self._raw_label_to_cp(
            self._cp_raw[local_idx],
            self._is_mate[local_idx],
            self._mate_in[local_idx]
        )
        y_value = self._cp_to_target(cp)

        y_policy = np.int64(self._policy_index[local_idx])
        legal_mask = self._legal_mask[local_idx].astype(np.bool_, copy=False)

        return (
            torch.from_numpy(np.array(x, copy=True)),
            torch.tensor([y_value], dtype=torch.float32),
            torch.tensor(y_policy, dtype=torch.long),
            torch.from_numpy(np.array(legal_mask, copy=True)),
        )




In [4]:
from pathlib import Path
import torch
from torch.utils.data import DataLoader, random_split

BATCH_SIZE = 1024
VAL_FRAC = 0.1

MANIFEST_PATH = Path("../data/fast_shards_rawcp_shuffled/manifest.json")
in_ch = 19

full_ds = FastChessShardDataset(
    MANIFEST_PATH,
    flatten=False,
    cp_clip=3000.0,
    sigmoid_scale=400.0,
    mate_cp=10000.0,
    use_sigmoid=True,
)

class ShardRangeDataset(Dataset):
    def __init__(self, base_ds, start_idx, end_idx):
        self.base_ds = base_ds
        self.start_idx = int(start_idx)
        self.end_idx = int(end_idx)

    def __len__(self):
        return self.end_idx - self.start_idx

    def __getitem__(self, idx):
        return self.base_ds[self.start_idx + idx]


N = len(full_ds)

num_shards = len(full_ds.lengths)
val_shards = max(1, int(round(num_shards * VAL_FRAC)))
train_shards = num_shards - val_shards

train_len = sum(full_ds.lengths[:train_shards])
val_len = sum(full_ds.lengths[train_shards:])

train_ds = ShardRangeDataset(full_ds, 0, train_len)
val_ds = ShardRangeDataset(full_ds, train_len, train_len + val_len)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
)
print(f"Dataset: N={N} | input_channels={in_ch} | train={train_len} | val={val_len}")

xb0, y_value0, y_policy0, legal_mask0 = next(iter(train_loader))
print("xb0:", xb0.shape, xb0.dtype)
print("y_value0:", y_value0.shape, y_value0.dtype)
print("y_policy0:", y_policy0.shape, y_policy0.dtype)
print("legal_mask0:", legal_mask0.shape, legal_mask0.dtype)
print("policy target legal?:", bool(legal_mask0[0, y_policy0[0]].item()))

Dataset: N=16492490 | input_channels=19 | train=14750000 | val=1742490
xb0: torch.Size([1024, 19, 8, 8]) torch.float32
y_value0: torch.Size([1024, 1]) torch.float32
y_policy0: torch.Size([1024]) torch.int64
legal_mask0: torch.Size([1024, 4672]) torch.bool
policy target legal?: True


In [5]:
xb0, y_value0, y_policy0, legal_mask0 = next(iter(train_loader))

print("xb0:", xb0.shape, xb0.dtype)
print("y_value0:", y_value0.shape, y_value0.dtype)
print("y_policy0:", y_policy0.shape, y_policy0.dtype)
print("legal_mask0:", legal_mask0.shape, legal_mask0.dtype)

assert xb0.shape[1] == in_ch, f"Expected {in_ch} channels, got {xb0.shape[1]}"
assert legal_mask0.shape[1] == 4672, f"Expected 4672 actions, got {legal_mask0.shape[1]}"
assert y_policy0.dim() == 1, f"Expected y_policy shape (B,), got {y_policy0.shape}"

xb_min, xb_max = float(xb0.min()), float(xb0.max())
print(f"X range first batch: [{xb_min}, {xb_max}]")

print("y_value batch stats: min=", float(y_value0.min()), "max=", float(y_value0.max()))
print("legal moves in first sample:", int(legal_mask0[0].sum()))
print("policy target legal?:", bool(legal_mask0[0, y_policy0[0]].item()))

def has_bad(t):
    return torch.isnan(t).any().item() or torch.isinf(t).any().item()

for i, (xchk, yvchk, ypchk, lmchk) in enumerate(train_loader):
    if i == 3:
        break
    assert not has_bad(xchk), "NaN/Inf in inputs"
    assert not has_bad(yvchk), "NaN/Inf in value targets"
    assert not has_bad(lmchk.float()), "NaN/Inf in legal masks"
    assert ypchk.dtype == torch.int64, f"Policy dtype should be int64, got {ypchk.dtype}"
    assert lmchk.shape[1] == 4672, f"Bad legal mask shape: {lmchk.shape}"
    idx = torch.arange(ypchk.shape[0])
    assert lmchk[idx, ypchk].all().item(), "Some policy targets are illegal"

print("Basic data checks passed.")

xb0: torch.Size([1024, 19, 8, 8]) torch.float32
y_value0: torch.Size([1024, 1]) torch.float32
y_policy0: torch.Size([1024]) torch.int64
legal_mask0: torch.Size([1024, 4672]) torch.bool
X range first batch: [-1.0, 1.0]
y_value batch stats: min= 0.000552778656128794 max= 0.999447226524353
legal moves in first sample: 49
policy target legal?: True
Basic data checks passed.


In [6]:
EPOCHS   = 120
PATIENCE = 10

FILTERS     = 128
NUM_BLOCKS  = 10
LR          = 1e-3
WEIGHT_DECAY = 1e-4
SEED = 42
PRINT_EVERY = 200
VALIDATE_EVERY = 1



import random
import numpy as np
import torch

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ResBlock(nn.Module):
    def __init__(self, channels=128):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x, inplace=True)

        x = self.conv2(x)
        x = self.bn2(x)

        x = x + residual
        x = F.relu(x, inplace=True)
        return x


class ChessNet(nn.Module):
    def __init__(self, in_ch=19, filters=128, num_blocks=10, policy_dim=4672):
        super().__init__()

        # shared trunk
        self.input_conv = nn.Conv2d(in_ch, filters, kernel_size=3, padding=1, bias=False)
        self.input_bn   = nn.BatchNorm2d(filters)

        self.res_blocks = nn.Sequential(*[ResBlock(filters) for _ in range(num_blocks)])

        # value head
        self.value_conv = nn.Conv2d(filters, 1, kernel_size=1, bias=False)
        self.value_bn   = nn.BatchNorm2d(1)
        self.value_fc1  = nn.Linear(64, 64)
        self.value_fc2  = nn.Linear(64, 1)

        # policy head
        self.policy_conv = nn.Conv2d(filters, 2, kernel_size=1, bias=False)
        self.policy_bn   = nn.BatchNorm2d(2)
        self.policy_fc   = nn.Linear(2 * 8 * 8, policy_dim)

    def forward_features(self, x):
        x = self.input_conv(x)
        x = self.input_bn(x)
        x = F.relu(x, inplace=True)
        x = self.res_blocks(x)
        return x

    def forward_raw(self, x, legal_mask=None):
        x = self.forward_features(x)

        # value head
        v = self.value_conv(x)
        v = self.value_bn(v)
        v = F.relu(v, inplace=True)
        v = v.view(v.size(0), -1)     # (B, 64)
        v = F.relu(self.value_fc1(v), inplace=True)
        v = self.value_fc2(v)         # raw scalar logit-like value

        # policy head
        p = self.policy_conv(x)
        p = self.policy_bn(p)
        p = F.relu(p, inplace=True)
        p = p.view(p.size(0), -1)     # (B, 128)
        p = self.policy_fc(p)         # raw logits, (B, 4672)

        if legal_mask is not None:
            p = p.masked_fill(~legal_mask, float("-inf"))

        return v, p

    def forward(self, x, legal_mask=None):
        v, p = self.forward_raw(x, legal_mask)
        v = torch.sigmoid(v)
        p = F.softmax(p, dim=-1)
        return v, p

In [8]:
value_criterion = nn.MSELoss()
policy_criterion = nn.CrossEntropyLoss()

In [9]:
model = ChessNet(
    in_ch=19,
    filters=FILTERS,
    num_blocks=NUM_BLOCKS,
    policy_dim=4672
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

Nparams = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Params: {Nparams/1e6:.3f}M")

Params: 3.584M


In [10]:
model.train()
xb = xb0.to(device)

dummy_legal_mask = torch.ones((xb.size(0), 4672), dtype=torch.bool, device=device)

v_out, p_out = model(xb, dummy_legal_mask)

print("value output:", v_out.shape)   # (B, 1)
print("policy output:", p_out.shape)  # (B, 4672)

value output: torch.Size([1024, 1])
policy output: torch.Size([1024, 4672])


In [11]:
model.train()

xb, y_value, y_policy, legal_mask = next(iter(train_loader))

xb = xb.to(device)
y_value = y_value.to(device)
y_policy = y_policy.to(device).long()
legal_mask = legal_mask.to(device)

v_pred, p_logits = model.forward_raw(xb, legal_mask)
v_pred_sigmoid = torch.sigmoid(v_pred)

loss_v = value_criterion(v_pred_sigmoid, y_value)
loss_p = policy_criterion(p_logits, y_policy)
loss = loss_v + loss_p

optimizer.zero_grad(set_to_none=True)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()

print("loss_v:", float(loss_v))
print("loss_p:", float(loss_p))
print("loss:", float(loss))

loss_v: 0.02713118866086006
loss_p: 3.277395486831665
loss: 3.3045265674591064


In [11]:
# ---- Fast tiny overfit test: preload tiny subset into RAM ----

from torch.utils.data import TensorDataset, DataLoader, Subset
import torch
import torch.nn as nn

small_idx = list(range(min(128, len(train_ds))))   # use 128 first, not 1024

tmp_loader = DataLoader(
    Subset(train_ds, small_idx),
    batch_size=128,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

xs, yvs, yps, lms = [], [], [], []

for xb, y_value, y_policy, legal_mask in tmp_loader:
    xs.append(xb)
    yvs.append(y_value)
    yps.append(y_policy)
    lms.append(legal_mask)

X_small = torch.cat(xs, dim=0)
Yv_small = torch.cat(yvs, dim=0)
Yp_small = torch.cat(yps, dim=0)
Lm_small = torch.cat(lms, dim=0)

tiny_ds_ram = TensorDataset(X_small, Yv_small, Yp_small, Lm_small)
tiny_loader = DataLoader(
    tiny_ds_ram,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

print("Tiny RAM dataset:")
print(X_small.shape, Yv_small.shape, Yp_small.shape, Lm_small.shape)

Tiny RAM dataset:
torch.Size([128, 19, 8, 8]) torch.Size([128, 1]) torch.Size([128]) torch.Size([128, 4672])


In [12]:
model_small = ChessNet(
    in_ch=in_ch,
    filters=64,      # smaller for debugging
    num_blocks=2,    # much smaller for overfit sanity check
    policy_dim=4672
).to(device)

opt_small = torch.optim.Adam(
    model_small.parameters(),
    lr=1e-3,
    betas=(0.9, 0.99),
    eps=1e-8
)

value_crit = nn.MSELoss()
policy_crit = nn.CrossEntropyLoss()

print("Starting tiny overfit test...")

for e in range(200):
    model_small.train()
    tl = 0.0
    tl_v = 0.0
    tl_p = 0.0
    correct = 0
    total = 0

    for xb, y_value, y_policy, legal_mask in tiny_loader:
        xb = xb.to(device)
        y_value = y_value.to(device)
        y_policy = y_policy.to(device)
        legal_mask = legal_mask.to(device)

        opt_small.zero_grad(set_to_none=True)

        v_raw, p_logits = model_small.forward_raw(xb, legal_mask)
        v_pred = torch.sigmoid(v_raw)

        loss_v = value_crit(v_pred, y_value)
        loss_p = policy_crit(p_logits, y_policy)
        loss = loss_v + loss_p

        loss.backward()
        opt_small.step()

        bs = xb.size(0)
        tl += loss.item() * bs
        tl_v += loss_v.item() * bs
        tl_p += loss_p.item() * bs

        pred_policy = p_logits.argmax(dim=1)
        correct += (pred_policy == y_policy).sum().item()
        total += bs

    print(
        f"(tiny) epoch {e+1:03d} "
        f"loss {tl/len(tiny_ds_ram):.6f} "
        f"value {tl_v/len(tiny_ds_ram):.6f} "
        f"policy {tl_p/len(tiny_ds_ram):.6f} "
        f"acc {correct/total:.4f}"
    )

Starting tiny overfit test...
(tiny) epoch 001 loss 3.191797 value 0.020767 policy 3.171030 acc 0.1094
(tiny) epoch 002 loss 2.726775 value 0.019098 policy 2.707677 acc 0.3516
(tiny) epoch 003 loss 2.326353 value 0.018637 policy 2.307716 acc 0.6719
(tiny) epoch 004 loss 1.949464 value 0.017496 policy 1.931968 acc 0.8828
(tiny) epoch 005 loss 1.577769 value 0.016480 policy 1.561289 acc 0.9609
(tiny) epoch 006 loss 1.231911 value 0.015474 policy 1.216437 acc 0.9766
(tiny) epoch 007 loss 0.959627 value 0.014391 policy 0.945235 acc 1.0000
(tiny) epoch 008 loss 0.744448 value 0.014231 policy 0.730217 acc 1.0000
(tiny) epoch 009 loss 0.564237 value 0.013062 policy 0.551175 acc 1.0000
(tiny) epoch 010 loss 0.426195 value 0.011652 policy 0.414543 acc 1.0000
(tiny) epoch 011 loss 0.338923 value 0.010348 policy 0.328575 acc 1.0000
(tiny) epoch 012 loss 0.273213 value 0.008923 policy 0.264290 acc 1.0000
(tiny) epoch 013 loss 0.214668 value 0.007301 policy 0.207367 acc 1.0000
(tiny) epoch 014 loss

In [14]:
ckpt = torch.load(CKPT_PATH, map_location=device)

model.load_state_dict(ckpt["model_state"])

if "optimizer_state" in ckpt:
    optimizer.load_state_dict(ckpt["optimizer_state"])
if "scaler_state" in ckpt:
    scaler.load_state_dict(ckpt["scaler_state"])
if "scheduler_state" in ckpt:
    scheduler.load_state_dict(ckpt["scheduler_state"])

start_epoch = ckpt.get("epoch", 0) + 1
best_val = ckpt.get("best_val", ckpt.get("best_val_loss", float("inf")))
pat = ckpt.get("pat", 0)

print(f"Resuming from epoch {start_epoch}, best_val={best_val:.6f}, pat={pat}")

Resuming from epoch 2, best_val=3.241858, pat=0


In [15]:
best_val = float("inf")
pat = 0
WV = 75

for epoch in range(1, EPOCHS + 1):
    model.train()

    train_loss = 0.0
    train_loss_v = 0.0
    train_loss_p = 0.0
    train_correct = 0
    train_total = 0
    num_train_batches = len(train_loader)

    for step, (xb, y_value, y_policy, legal_mask) in enumerate(train_loader, start=1):
        xb = xb.to(device, non_blocking=True)
        y_value = y_value.to(device, non_blocking=True)
        y_policy = y_policy.to(device, non_blocking=True)
        legal_mask = legal_mask.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            v_raw, p_logits = model.forward_raw(xb, legal_mask)
            v_pred = torch.sigmoid(v_raw)

            loss_v = value_criterion(v_pred, y_value)
            loss_p = policy_criterion(p_logits, y_policy)
            loss = WV*loss_v + loss_p

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        bs = xb.size(0)
        train_loss += loss.item() * bs
        train_loss_v += loss_v.item() * bs
        train_loss_p += loss_p.item() * bs

        pred_policy = p_logits.argmax(dim=1)
        train_correct += (pred_policy == y_policy).sum().item()
        train_total += bs

        if step % PRINT_EVERY == 0 or step == num_train_batches:
            print(
                f"Epoch {epoch:02d} | train step {step}/{num_train_batches} | "
                f"loss {loss.item():.6f} | v {loss_v.item():.6f} | p {loss_p.item():.6f}"
            )

    train_loss /= len(train_loader.dataset)
    train_loss_v /= len(train_loader.dataset)
    train_loss_p /= len(train_loader.dataset)
    train_acc = train_correct / train_total


    do_val = (epoch % VALIDATE_EVERY == 0)

    if do_val:
        model.eval()
        val_loss = 0.0
        val_loss_v = 0.0
        val_loss_p = 0.0
        val_correct = 0
        val_total = 0
        num_val_batches = len(val_loader)

        with torch.no_grad():
            for step, (xb, y_value, y_policy, legal_mask) in enumerate(val_loader, start=1):
                xb = xb.to(device, non_blocking=True)
                y_value = y_value.to(device, non_blocking=True)
                y_policy = y_policy.to(device, non_blocking=True)
                legal_mask = legal_mask.to(device, non_blocking=True)

                with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                    v_raw, p_logits = model.forward_raw(xb, legal_mask)
                    v_pred = torch.sigmoid(v_raw)

                    loss_v = value_criterion(v_pred, y_value)
                    loss_p = policy_criterion(p_logits, y_policy)
                    loss = WV*loss_v + loss_p

                bs = xb.size(0)
                val_loss += loss.item() * bs
                val_loss_v += loss_v.item() * bs
                val_loss_p += loss_p.item() * bs

                pred_policy = p_logits.argmax(dim=1)
                val_correct += (pred_policy == y_policy).sum().item()
                val_total += bs

                if step % PRINT_EVERY == 0 or step == num_val_batches:
                    print(
                        f"Epoch {epoch:02d} | val step {step}/{num_val_batches} | "
                        f"loss {loss.item():.6f} | v {loss_v.item():.6f} | p {loss_p.item():.6f}"
                    )

        val_loss /= len(val_loader.dataset)
        val_loss_v /= len(val_loader.dataset)
        val_loss_p /= len(val_loader.dataset)
        val_acc = val_correct / val_total

        scheduler.step()

        print(
            f"Epoch {epoch:02d} | "
            f"train loss: {train_loss:.6f} "
            f"(v {train_loss_v:.6f}, p {train_loss_p:.6f}, acc {train_acc:.4f}) | "
            f"val loss: {val_loss:.6f} "
            f"(v {val_loss_v:.6f}, p {val_loss_p:.6f}, acc {val_acc:.4f})"
        )

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            pat = 0
            torch.save(
                {
                    "model_state": model.state_dict(),
                    "input_channels": int(in_ch),
                    "filters": int(FILTERS),
                    "num_blocks": int(NUM_BLOCKS),
                    "policy_dim": 4672,
                    "best_val_loss": float(best_val),
                    "epoch": int(epoch),
                },
                CKPT_PATH,
            )
        else:
            pat += 1
            if pat >= PATIENCE:
                print(f"Early stopping at epoch {epoch}. Best val loss: {best_val:.6f}")
                break
    else:
        scheduler.step()
        print(
            f"Epoch {epoch:02d} | "
            f"train loss: {train_loss:.6f} "
            f"(v {train_loss_v:.6f}, p {train_loss_p:.6f}, acc {train_acc:.4f})"
        )

print("Best val loss:", best_val)
print(f"Saved best model to: {CKPT_PATH}")

Epoch 01 | train step 200/14405 | loss 3.232358 | v 0.012546 | p 2.291388
Epoch 01 | train step 400/14405 | loss 2.975935 | v 0.010282 | p 2.204818
Epoch 01 | train step 600/14405 | loss 3.136739 | v 0.011743 | p 2.256031
Epoch 01 | train step 800/14405 | loss 3.016419 | v 0.010179 | p 2.253007
Epoch 01 | train step 1000/14405 | loss 3.065767 | v 0.010200 | p 2.300786
Epoch 01 | train step 1200/14405 | loss 3.040708 | v 0.011654 | p 2.166625
Epoch 01 | train step 1400/14405 | loss 2.836388 | v 0.008857 | p 2.172088
Epoch 01 | train step 1600/14405 | loss 3.063085 | v 0.010334 | p 2.288014
Epoch 01 | train step 1800/14405 | loss 2.922668 | v 0.009933 | p 2.177695
Epoch 01 | train step 2000/14405 | loss 2.978287 | v 0.010368 | p 2.200688
Epoch 01 | train step 2200/14405 | loss 3.072330 | v 0.011455 | p 2.213170
Epoch 01 | train step 2400/14405 | loss 3.048838 | v 0.011069 | p 2.218628
Epoch 01 | train step 2600/14405 | loss 3.012644 | v 0.010060 | p 2.258173
Epoch 01 | train step 2800/14

KeyboardInterrupt: 

In [16]:
torch.save(
    {
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scaler_state": scaler.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "input_channels": int(in_ch),
        "filters": int(FILTERS),
        "num_blocks": int(NUM_BLOCKS),
        "policy_dim": 4672,
        "best_val_loss": float(best_val),
        "epoch": int(epoch),
        "pat": int(pat),
        "WV": float(WV),
    },
    CKPT_PATH,
)

In [17]:
# %% CONTINUATION CELL — fixed resume/checkpoint/validation

from pathlib import Path
import torch
import torch.nn as nn
import math

WV = 75
PATIENCE = 5
MAX_VAL_BATCHES = 200          # change to 100 if validation still too slow
MIN_DELTA = 1e-6

BEST_CKPT_PATH = Path(CKPT_PATH)
LAST_CKPT_PATH = Path(str(CKPT_PATH).replace(".pth", "_last.pth"))

# If we interrupted mid-epoch, restart that epoch.
# If no epoch exists, start at 1.
start_epoch = int(globals().get("epoch", 1))

# Keep previous best if it exists; otherwise use inf
best_val = float(globals().get("best_val", float("inf")))
pat = int(globals().get("pat", 0))

# Save current in-memory state before continuing
torch.save(
    {
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scaler_state": scaler.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "input_channels": int(in_ch),
        "filters": int(FILTERS),
        "num_blocks": int(NUM_BLOCKS),
        "policy_dim": 4672,
        "best_val_loss": float(best_val),
        "epoch": max(start_epoch - 1, 0),
        "pat": int(pat),
        "WV": float(WV),
    },
    LAST_CKPT_PATH,
)

print(f"Saved current state to {LAST_CKPT_PATH}")
print(f"Continuing from epoch {start_epoch}, best_val={best_val}, pat={pat}")


for epoch in range(start_epoch, EPOCHS + 1):
    model.train()

    train_loss = 0.0
    train_loss_v = 0.0
    train_loss_p = 0.0
    train_correct = 0
    train_total = 0

    num_train_batches = len(train_loader)

    for step, (xb, y_value, y_policy, legal_mask) in enumerate(train_loader, start=1):
        xb = xb.to(device, non_blocking=True)
        y_value = y_value.to(device, non_blocking=True)
        y_policy = y_policy.to(device, non_blocking=True)
        legal_mask = legal_mask.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            v_raw, p_logits = model.forward_raw(xb, legal_mask)
            v_pred = torch.sigmoid(v_raw)

            loss_v = value_criterion(v_pred, y_value)
            loss_p = policy_criterion(p_logits, y_policy)
            loss = WV * loss_v + loss_p

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        bs = xb.size(0)
        train_loss += loss.item() * bs
        train_loss_v += loss_v.item() * bs
        train_loss_p += loss_p.item() * bs

        pred_policy = p_logits.argmax(dim=1)
        train_correct += (pred_policy == y_policy).sum().item()
        train_total += bs

        if step % PRINT_EVERY == 0 or step == num_train_batches:
            print(
                f"Epoch {epoch:03d} | train step {step}/{num_train_batches} | "
                f"loss {loss.item():.6f} | v {loss_v.item():.6f} | p {loss_p.item():.6f}"
            )

    train_loss /= train_total
    train_loss_v /= train_total
    train_loss_p /= train_total
    train_acc = train_correct / train_total

    # ---- partial validation ----
    model.eval()

    val_loss = 0.0
    val_loss_v = 0.0
    val_loss_p = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for step, (xb, y_value, y_policy, legal_mask) in enumerate(val_loader, start=1):
            if step > MAX_VAL_BATCHES:
                break

            xb = xb.to(device, non_blocking=True)
            y_value = y_value.to(device, non_blocking=True)
            y_policy = y_policy.to(device, non_blocking=True)
            legal_mask = legal_mask.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                v_raw, p_logits = model.forward_raw(xb, legal_mask)
                v_pred = torch.sigmoid(v_raw)

                loss_v = value_criterion(v_pred, y_value)
                loss_p = policy_criterion(p_logits, y_policy)
                loss = WV * loss_v + loss_p

            bs = xb.size(0)
            val_loss += loss.item() * bs
            val_loss_v += loss_v.item() * bs
            val_loss_p += loss_p.item() * bs

            pred_policy = p_logits.argmax(dim=1)
            val_correct += (pred_policy == y_policy).sum().item()
            val_total += bs

    val_loss /= val_total
    val_loss_v /= val_total
    val_loss_p /= val_total
    val_acc = val_correct / val_total

    scheduler.step()

    print(
        f"Epoch {epoch:03d} | "
        f"train {train_loss:.6f} "
        f"(v {train_loss_v:.6f}, p {train_loss_p:.6f}, acc {train_acc:.4f}) | "
        f"val {val_loss:.6f} "
        f"(v {val_loss_v:.6f}, p {val_loss_p:.6f}, acc {val_acc:.4f}) | "
        f"lr {optimizer.param_groups[0]['lr']:.2e}"
    )

    # Always save last checkpoint
    torch.save(
        {
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scaler_state": scaler.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "input_channels": int(in_ch),
            "filters": int(FILTERS),
            "num_blocks": int(NUM_BLOCKS),
            "policy_dim": 4672,
            "best_val_loss": float(best_val),
            "epoch": int(epoch),
            "pat": int(pat),
            "WV": float(WV),
        },
        LAST_CKPT_PATH,
    )

    # Save best checkpoint
    if val_loss < best_val - MIN_DELTA:
        best_val = val_loss
        pat = 0

        torch.save(
            {
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scaler_state": scaler.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "input_channels": int(in_ch),
                "filters": int(FILTERS),
                "num_blocks": int(NUM_BLOCKS),
                "policy_dim": 4672,
                "best_val_loss": float(best_val),
                "epoch": int(epoch),
                "pat": int(pat),
                "WV": float(WV),
            },
            BEST_CKPT_PATH,
        )

        print(f"Saved new best: {BEST_CKPT_PATH} | best_val={best_val:.6f}")

    else:
        pat += 1
        print(f"No improvement. pat={pat}/{PATIENCE}")

        if pat >= PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best val loss: {best_val:.6f}")
            break

print("Best val loss:", best_val)
print("Best checkpoint:", BEST_CKPT_PATH)
print("Last checkpoint:", LAST_CKPT_PATH)

Saved current state to ..\weights\new_labels2_last.pth
Continuing from epoch 3, best_val=2.8993325929944, pat=0
Epoch 003 | train step 200/14405 | loss 2.814265 | v 0.009062 | p 2.134641
Epoch 003 | train step 400/14405 | loss 2.750955 | v 0.008748 | p 2.094879
Epoch 003 | train step 600/14405 | loss 2.823956 | v 0.009564 | p 2.106669
Epoch 003 | train step 800/14405 | loss 2.865632 | v 0.009119 | p 2.181734
Epoch 003 | train step 1000/14405 | loss 2.999880 | v 0.010260 | p 2.230350
Epoch 003 | train step 1200/14405 | loss 2.826562 | v 0.008116 | p 2.217850
Epoch 003 | train step 1400/14405 | loss 2.891862 | v 0.010387 | p 2.112821
Epoch 003 | train step 1600/14405 | loss 2.720277 | v 0.008303 | p 2.097523
Epoch 003 | train step 1800/14405 | loss 2.967455 | v 0.011090 | p 2.135694
Epoch 003 | train step 2000/14405 | loss 2.715405 | v 0.007983 | p 2.116711
Epoch 003 | train step 2200/14405 | loss 2.850347 | v 0.008898 | p 2.182996
Epoch 003 | train step 2400/14405 | loss 2.799908 | v 0.

In [ ]:
# # convert_npz_shards_to_fast.py

# from pathlib import Path
# import json
# import numpy as np

# SRC_DIR = Path("../data/npz_shards_rawcp")
# DST_DIR = Path("../data/fast_shards_rawcp")
# DST_DIR.mkdir(parents=True, exist_ok=True)

# src_files = sorted(SRC_DIR.glob("shard_*.npz"))
# assert src_files, f"No shards found in {SRC_DIR}"

# manifest = {
#     "source_dir": str(SRC_DIR.resolve()),
#     "dest_dir": str(DST_DIR.resolve()),
#     "num_shards": 0,
#     "total_samples": 0,
#     "shards": []
# }

# for i, src in enumerate(src_files):
#     print(f"[{i+1}/{len(src_files)}] converting {src.name}")

#     with np.load(src, allow_pickle=True) as zf:
#         positions = zf["positions"].astype(np.float32, copy=False)
#         cp_raw = zf["evaluations_cp_raw"].astype(np.int32, copy=False)
#         is_mate = zf["is_mate"].astype(np.int8, copy=False)
#         mate_in = zf["mate_in"].astype(np.int16, copy=False)
#         policy_index = zf["policy_index"].astype(np.int32, copy=False)
#         legal_mask = zf["legal_mask"].astype(np.bool_, copy=False)

#     shard_dir = DST_DIR / src.stem
#     shard_dir.mkdir(parents=True, exist_ok=True)

#     np.save(shard_dir / "positions.npy", positions, allow_pickle=False)
#     np.save(shard_dir / "evaluations_cp_raw.npy", cp_raw, allow_pickle=False)
#     np.save(shard_dir / "is_mate.npy", is_mate, allow_pickle=False)
#     np.save(shard_dir / "mate_in.npy", mate_in, allow_pickle=False)
#     np.save(shard_dir / "policy_index.npy", policy_index, allow_pickle=False)
#     np.save(shard_dir / "legal_mask.npy", legal_mask, allow_pickle=False)

#     n = int(positions.shape[0])
#     manifest["num_shards"] += 1
#     manifest["total_samples"] += n
#     manifest["shards"].append({
#         "name": src.stem,
#         "dir": str(shard_dir.resolve()),
#         "length": n
#     })

# manifest_path = DST_DIR / "manifest.json"
# with open(manifest_path, "w", encoding="utf-8") as f:
#     json.dump(manifest, f, indent=2)

# print("\nDone.")
# print("Saved manifest to:", manifest_path)
# print("Num shards:", manifest["num_shards"])
# print("Total samples:", manifest["total_samples"])

[1/3299] converting shard_00000.npz
[2/3299] converting shard_00001.npz
[3/3299] converting shard_00002.npz
[4/3299] converting shard_00003.npz
[5/3299] converting shard_00004.npz
[6/3299] converting shard_00005.npz
[7/3299] converting shard_00006.npz
[8/3299] converting shard_00007.npz
[9/3299] converting shard_00008.npz
[10/3299] converting shard_00009.npz
[11/3299] converting shard_00010.npz
[12/3299] converting shard_00011.npz
[13/3299] converting shard_00012.npz
[14/3299] converting shard_00013.npz
[15/3299] converting shard_00014.npz
[16/3299] converting shard_00015.npz
[17/3299] converting shard_00016.npz
[18/3299] converting shard_00017.npz
[19/3299] converting shard_00018.npz
[20/3299] converting shard_00019.npz
[21/3299] converting shard_00020.npz
[22/3299] converting shard_00021.npz
[23/3299] converting shard_00022.npz
[24/3299] converting shard_00023.npz
[25/3299] converting shard_00024.npz
[26/3299] converting shard_00025.npz
[27/3299] converting shard_00026.npz
[28/3299] 

In [17]:
import torch
import numpy as np

def shared_trunk_grad_norm(model):
    sq = 0.0
    for name, p in model.named_parameters():
        if p.grad is None:
            continue
        if (
            name.startswith("input_conv")
            or name.startswith("input_bn")
            or name.startswith("res_blocks")
        ):
            g = p.grad.detach()
            sq += g.pow(2).sum().item()
    return sq ** 0.5


def measure_head_gradient_balance(
    model,
    loader,
    value_criterion,
    policy_criterion,
    device,
    num_batches=20,
    print_each_batch=True,
):
    model.train()

    gv_list = []
    gp_list = []
    lv_list = []
    lp_list = []

    it = iter(loader)

    for batch_idx in range(num_batches):
        try:
            xb, y_value, y_policy, legal_mask = next(it)
        except StopIteration:
            break

        xb = xb.to(device, non_blocking=True)
        y_value = y_value.to(device, non_blocking=True)
        y_policy = y_policy.to(device, non_blocking=True)
        legal_mask = legal_mask.to(device, non_blocking=True)

        # forward once in full precision for clean diagnostics
        model.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", enabled=False):
            xb_fp = xb.float()
            y_value_fp = y_value.float()

            v_raw, p_logits = model.forward_raw(xb_fp, legal_mask)
            v_pred = torch.sigmoid(v_raw)

            loss_v = value_criterion(v_pred, y_value_fp)
            loss_p = policy_criterion(p_logits, y_policy)

        # gradients from value head only
        model.zero_grad(set_to_none=True)
        loss_v.backward(retain_graph=True)
        gv = shared_trunk_grad_norm(model)

        # gradients from policy head only
        model.zero_grad(set_to_none=True)
        loss_p.backward()
        gp = shared_trunk_grad_norm(model)

        lv = loss_v.item()
        lp = loss_p.item()

        gv_list.append(gv)
        gp_list.append(gp)
        lv_list.append(lv)
        lp_list.append(lp)

        if print_each_batch:
            ratio = gp / (gv + 1e-12)
            print(
                f"batch {batch_idx+1:02d} | "
                f"loss_v {lv:.6f} | loss_p {lp:.6f} | "
                f"grad_v {gv:.6e} | grad_p {gp:.6e} | "
                f"gp/gv {ratio:.3f}"
            )

    if len(gv_list) == 0:
        print("No batches were processed.")
        return None

    mean_lv = float(np.mean(lv_list))
    mean_lp = float(np.mean(lp_list))
    mean_gv = float(np.mean(gv_list))
    mean_gp = float(np.mean(gp_list))

    ratio_gp_gv = mean_gp / (mean_gv + 1e-12)

    suggested_equal = ratio_gp_gv
    suggested_value_2x = 2.0 * ratio_gp_gv
    suggested_value_3x = 3.0 * ratio_gp_gv

    print("\n===== SUMMARY =====")
    print(f"avg loss_v      : {mean_lv:.6f}")
    print(f"avg loss_p      : {mean_lp:.6f}")
    print(f"avg grad_v trunk: {mean_gv:.6e}")
    print(f"avg grad_p trunk: {mean_gp:.6e}")
    print(f"avg gp/gv       : {ratio_gp_gv:.3f}")
    print()
    print(f"Suggested w_v for equal trunk influence   : {suggested_equal:.3f}")
    print(f"Suggested w_v for value 2x stronger       : {suggested_value_2x:.3f}")
    print(f"Suggested w_v for value 3x stronger       : {suggested_value_3x:.3f}")
    print("Assumes w_p = 1.0")

    return {
        "avg_loss_v": mean_lv,
        "avg_loss_p": mean_lp,
        "avg_grad_v": mean_gv,
        "avg_grad_p": mean_gp,
        "avg_gp_over_gv": ratio_gp_gv,
        "suggested_wv_equal": suggested_equal,
        "suggested_wv_value_2x": suggested_value_2x,
        "suggested_wv_value_3x": suggested_value_3x,
    }

In [18]:
stats = measure_head_gradient_balance(
    model=model,
    loader=train_loader,
    value_criterion=value_criterion,
    policy_criterion=policy_criterion,
    device=device,
    num_batches=20,
    print_each_batch=True,
)

batch 01 | loss_v 0.005352 | loss_p 2.179014 | grad_v 5.050454e-02 | grad_p 1.634523e+00 | gp/gv 32.364
batch 02 | loss_v 0.006757 | loss_p 2.186065 | grad_v 7.406083e-02 | grad_p 1.509221e+00 | gp/gv 20.378
batch 03 | loss_v 0.005757 | loss_p 2.284953 | grad_v 5.487177e-02 | grad_p 1.470044e+00 | gp/gv 26.791
batch 04 | loss_v 0.008631 | loss_p 2.164963 | grad_v 8.396015e-02 | grad_p 1.479663e+00 | gp/gv 17.623
batch 05 | loss_v 0.007415 | loss_p 2.224373 | grad_v 5.582631e-02 | grad_p 1.395983e+00 | gp/gv 25.006
batch 06 | loss_v 0.007704 | loss_p 2.141330 | grad_v 6.345184e-02 | grad_p 1.357134e+00 | gp/gv 21.388
batch 07 | loss_v 0.007717 | loss_p 2.259528 | grad_v 8.956599e-02 | grad_p 1.425769e+00 | gp/gv 15.919
batch 08 | loss_v 0.008659 | loss_p 2.145212 | grad_v 4.883902e-02 | grad_p 1.338235e+00 | gp/gv 27.401
batch 09 | loss_v 0.005259 | loss_p 2.218634 | grad_v 3.580035e-02 | grad_p 1.673600e+00 | gp/gv 46.748
batch 10 | loss_v 0.004724 | loss_p 2.124137 | grad_v 4.212301e-

In [19]:
WV = stats["suggested_wv_value_2x"]   # or _equal or _value_3x
WP = 1.0
print("Using WV =", WV, "WP =", WP)

Using WV = 51.20704472178229 WP = 1.0
